<a href="https://colab.research.google.com/github/yjenny2512/My-personal-repository/blob/main/huggingface_text_classiftication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learning Hugging Face Text Classification Tutorial

* Resource notebook: https://www.learnhuggingface.com/notebooks/hugging_face_text_classification_tutorial

Note : GPU is needed in Google Colab


### 1. Import necessary libraries


In [ ]:
# Install dependancies (this is mostly for Google Colab )

try :
  import datasets,evaluate,accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio
  import datasets,evaluate,accelerate
  import gradio as gr

import random
import numpy as np
import pandas as pd
import torch
import transformers

print(f"Using transformers version{transformers.__version__}")
print(f"Using torch version{torch.__version__}")
print(f"Using datasets version{datasets.__version__}")


## 2. Getting a dataset

Building food not food text classification model : need food not food text dataset



In [ ]:
from datasets import load_dataset

dataset = load_dataset("mrdbourke/learn_hf_food_not_food_image_captions")
dataset

In [ ]:
# What features are there?
dataset.column_names

In [ ]:
# Access the training split
dataset["train"][10]

In [ ]:
### Inspect random samples

random_indeces = random.sample(range(len(dataset["train"])),5)
random_indeces

random_samples = dataset["train"][random_indeces]
print(f"[INFO] Random samples from dataset:\n)")
for text,label in zip(random_samples["text"],random_samples["label"]):
  print(f"Text : {text} | Label : {label}")

In [ ]:
# Get unique label values
dataset["train"].unique("label")

In [ ]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])

In [ ]:
# Turn our dataset into a DataFrame and get a random sample

food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.head(7)

In [ ]:
food_not_food_df["label"].value_counts()

## 3. Preparing data for text classification

We want to :
1. Tokenize our text - turn our text into numbers
2. Create a train/test split - want to train our model on the training set and evaluate on the test set


In [ ]:
# Create a mapping for labels to numeric values
id2label = {0 : "not_food", 1:"food"}
label2id = {"not_food":0,"food":1}
print(id2label)
print(label2id)

In [ ]:
# Create mapping programmatically
id2label = {idx : label for idx,label in enumerate(dataset["train"].unique("label")[::-1]) }
label2id = {label : idx for idx,label in id2label.items()}
print(id2label )
print(label2id)

In [ ]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
  print(idx,label)
  id2label[idx] = label

In [ ]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample = {"text": "This is a sentence about my favourite food:honey","label":"food"}

# Test our function
map_labels_to_number(example_sample)

In [ ]:
# Map our dataset labels to numbers (the whole thing)
# We can do this with dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

In [ ]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

### 4. Splitting data into training and test sets

* Train set = model will learn patterns on this dataset
* Test set = model will evaluate patterns on this dataset

We can split our dataset using `datasets.Dataset.train_test_split´


In [ ]:
# Split our dataset into train/test split
dataset = dataset.train_test_split(test_size=0.2,seed=42)
dataset

In [ ]:
random_idx_train = random.randint(0,len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train



In [ ]:
random_idx_test = random.randint(0,len(dataset["test"]))
random_sample_test = dataset["test"][random_idx_test]
random_sample_test

### Tokenizing our text data(turning text into numbers)
The premise of tokenization is to turn words into numbers

___

The `transformers` library has in-built support for the HuggingFace tokenizers

And the class `transformers.AutoTokenizer` helps pair a model to a tokenizer

* Model/tokenizer we're going to use : distilbert/distilbert-base-uncased

* All models :https://huggingface.co/models?sort=trending

* Models are often paired with tokenizers
* Tokenizer = turn text into numbers
* Model = find patterns in those numbers


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True)#use the fast implementation (on by default , this requires Rust installed)

tokenizer

In [ ]:
# Test out the to")kenizer
tokenizer("I love pizza")

* input_ids = our text turned into numbers
* attention_mask = whether or not to pay attention to certain tokens

In [ ]:
tokenizer("I love pizza!")

In [ ]:
# Get the length of our tokenizer vocab
length_of_tokenizer_vocab = len(tokenizer.vocab)
print(f"[Info] Number of items in our tokenizer vocabulary: {length_of_tokenizer_vocab}")

# Get the maximum sequence length the tokenizer can handle
max_tokenizer_input_sequence_length = tokenizer.model_max_length
print(f"[Info] Max tokenizer input sequence length:{max_tokenizer_input_sequence_length}")


In [ ]:
tokenizer("akash")

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer("akash").input_ids)

In [ ]:
tokenizer.convert_ids_to_tokens(tokenizer("🍔").input_ids)


In [ ]:
# Get the first 5 items in the tokenizer vocab
sorted(tokenizer.vocab.items())[:5]

In [ ]:
import random
random.sample(sorted(tokenizer.vocab.items()),5)

### Making a preprocessing function to tokenize text

Want to make it easy to go from sample to tokenized sample


In [ ]:
def tokenize_text(examples):
  """
  Tokenize given example text and return the
  tokenized text
  """

  return tokenizer(examples["text"],
                   padding=True,#pad short sequences to the longest sequence length in batch
                   truncation=True)#truncate long sequences to the maximum length the model can handle




In [ ]:
example_sample_2 = {"text":"I love pizza","label":1}

# Test the function
tokenize_text(example_sample_2)

In [ ]:
long_text = "I love pizza " * 1000
len(long_text)

In [ ]:
tokenized_long_text = tokenize_text({"text": long_text,label : 1})
len(tokenized_long_text["input_ids"])

In [ ]:
dataset

In [ ]:
# Map our tokenite-text function to the dataset
tokenized_dataset = dataset.map(function=tokenize_text,
                               batched = True , # set batched=True to tokenize across batches of samples at a time rather than one at a time
                                batch_size=1000)
tokenized_dataset


**Note:** In ML it is often faster to do things in batches rather than one at a time due to leveraging computer hardware parallelization

In [ ]:
tokenizer.all_special_tokens

In [ ]:
tokenizer.all_special_ids

In [ ]:
# Get 2 samples from the tokenized datasets
train_tokenized_sample = tokenized_dataset["train"][0]
test_tokenized_sample = tokenized_dataset["test"][0]

for key in train_tokenized_sample.keys():
  print(f"[Info] Key :{key}")
  print(f"Train sample: {train_tokenized_sample[key]}")
  print(f"Test sample: {test_tokenized_sample[key]}")

### Tokenization takeaways

1. Tokenization = turn data into numbers (e.g. text -> map to number)
2. Many models are out there and have different tokenizers, Hugging Face's `Auto` (e.g. `AutoTokenizer`,`AutoProcessor`,`AutoModel`...) help to match tokenizers to models
3. Tokenization can happen in parallel using `map` and batched functions


## Setting up an evaluation metric

What we want to do is : use the evaluation metric to get a numerical idea of how our model is performing

Some common evaluation metrics for classifiction :

-Accuracy(how many examples out of 100 did you get correct?)
-Precision
-Recall
-F1 score

Evaluation metric is important because some projects may have an evaluation threshold you need to fulfill

Some places for evaluation metrics:


- Scikit-learn documentation : https://scikit-learn.org/stable/modules/model_evaluation.html

- Hugging Face evaluate : https://huggingface.co/docs/evaluate/index


In [ ]:
import evaluate
import numpy as np
from typing import Tuple

accuracy_metric = evaluate.load("accuracy")

def compute_accuracy(predictions_and_labels: Tuple[np.array,np.array]):
  """
  Computes the accuracy of a model by comparing the
  predictions and labels
  """
  predictions,labels = predictions_and_labels

  if len(predictions.shape) >= 2 :
    predictions = np.argmax(predictions,axis=1)

  return accuracy_metric.compute(predictions=predictions,references=labels)


In [ ]:
# Example predictions and accuracy score
example_preds_all_correct = np.array([0,0,0,0,0,0,0,0,0,0])
example_preds_one_incorrect = np.array([0,0,0,0,1,0,0,0,0,0])
example_labels = np.array([0,0,0,0,0,0,0,0,0,0])

# test the function
print(f"Accuracy when all predictions are correct: {compute_accuracy((example_preds_all_correct,example_labels))} ")
print(f"Accuracy when one prediction is incorrect: {compute_accuracy((example_preds_one_incorrect,example_labels))} ")

## Setting up a model for training

* We're going to be using transfer learning.
* Transfer learning is a powerful technique,unique to deep learning models that enables us to use the  patterns one model has learned on another problem for our own problem

Workflow for training:
1. Create and preprocess data(we've actually done that already)
2. Define the model we'd like to use for our problem
3. Define training arguments for training our model `transformers.TrainingArguments`
* These are also known as "hyperparameters"=settings on your model that you can adjust
* Parameters=weights/patterns in the model,that get updated automatically
4. Pass `TrainingArguments` to an instance of `transformers.Trainer`
5. Train the model by calling `Trainer.train()`
6. Save the model to our local machine or to the Hugging Face Hub
7. Evaluate the trained model by making and inspecting predictions on the test data (and our own custom data)
8. Turn the model into a sharable demo

In [ ]:
id2label

In [ ]:
label2id

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
    num_labels=2, # classify into food/not_food
    id2label=id2label,
    label2id=label2id)


In [ ]:
model

Our model is comprised of the following parts :
1. `embeddings` - embeddings are a form of learned representation of tokens. So if tokens are a direct mapping from token to number,embeddings are a learned vector representation
2. `transformer` - our model architecture backbone,this has discovered patterns/relationships in the embeddings
3. `classifier` - we need to customize this layer to suit our problem

**Note:** If you get input errors from passing a sample to a model,make sure the sample you pass to the model is formatting in the same way the model was trained on.for example if your model uses a specific tokenizer,make sure to tokenize your text before passing it to the model

In [ ]:
model(**tokenized_dataset["train"][0])

### Count the parameters in our model

Weights/parameters = small numeric opportunities for a model to learn patterns in data

In [ ]:
def count_params(model):
  """
  Count the parameters of a PyTorch model
  """
  trainable_parameters = sum(param.numel() for param in model.parameters() if param.requires_grad)
  total_parameters = sum(param.numel() for param in model.parameters())

  return {"trainable_parameters" : trainable_parameters,"total_parameters": total_parameters}

count_params(model)


Looks like our model has around 67M parameters and all of them are trainable.

**Note:**
* Generally,the more parameters-the more capacity it has to learn
* If you want the best possible performance - generally more parameters is better
  * However more params require more compute time


### Create a folder/directory for saving models


In [ ]:
# Create model output directory

from pathlib import Path

# Create models directory
models_dir = Path("models")
models_dir.mkdir(parents=True,exist_ok=True)

#  Create model save name
model_save_name = "learn-hf-food-not_food-text-classification-model-distilbert-base-uncased"

# Create model save path
model_save_dir = Path(models_dir,model_save_name)

model_save_dir

### Setting up hyperparameters with TrainingArguments

In [ ]:
from transformers import TrainingArguments

print(f"[Info] Saving model checkpoints: {model_save_dir} ")

#Create training arguments
training_args = TrainingArguments(
    output_dir=model_save_dir,
    learning_rate=0.0001,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    use_cpu=False,
    seed=42,
    load_best_model_at_end=True,
    logging_strategy="epoch",
    report_to="none",
    hub_private_repo=False
)

### Setting up an instance of Trainer

In [ ]:
from transformers import Trainer

# Setup Trainer
trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_accuracy
)

trainer

### Train the model by calling trainer.train()

In [ ]:
results = trainer.train()

In [ ]:
# Inspect training metrics
for key,value in results.metrics.items():
  print(f"{key} : {value}")

### Save the model for later use

**Note:** If you are saving a model to Google Colab, it will disappear from your Colab instance when it disconnects

In [ ]:
# Save the model
print(f"[Info] Saving model to {model_save_dir}")
trainer.save_model(output_dir=model_save_dir)

### Inspect the model training metrics

In [ ]:
# Get training history
trainer_history_all = trainer.state.log_history
trainer_history_metrics = trainer_history_all[:-1]
trainer_history_training_time=trainer_history_all[-1]

trainer_history_metrics[:3]

In [ ]:
import pprint

# Extract eval and training metrics
trainer_history_training_set = []
trainer_history_eval_set = []

# Loop through our metrics
for item in trainer_history_metrics:
  item_keys = list(item.keys())
  if any("eval" in item for item in item_keys):
    trainer_history_eval_set.append(item)
  else:
    trainer_history_training_set.append(item)

  # Show the first from each
  print(f"First item in training set:")
  pprint.pprint(trainer_history_training_set[:2])
  print(f"First item in eval set:")
  pprint.pprint(trainer_history_eval_set[:2])

### Taking a look at the loss curves

Loss curves = a good visualization of your model's performance over time

Ideally,loss curves will trend downwards


In [ ]:
# Create a pandas DataFrame for t6he training and evaluation metrics

trainer_history_train_df = pd.DataFrame(trainer_history_training_set)
trainer_history_eval_df = pd.DataFrame(trainer_history_eval_set)

trainer_history_train_df.head()

In [ ]:
# Plot the loss curves
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.plot(trainer_history_train_df["epoch"],trainer_history_train_df["loss"],label="Training loss")
plt.plot(trainer_history_eval_df["epoch"],trainer_history_eval_df["eval_loss"],label="Evaluation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Text classification fine-tuning Distilbert training and evaluation loss over time")
plt.legend()
plt.show()

### Pushing our model to the HF hub


In [ ]:
# Save our model to the HuggingFace hub



model_upload_url = trainer.push_to_hub(
    commit_message="Uploading food not food text classifier model"

)
print(f"Model successfully uploaded to the HuggingFace hub with URL:{model_upload_url}")

### Making and evaluating predictions on  the test data
**Note:** Evaluating a model is just as important as training a model

In [ ]:
# Perform predictions on the test data
predictions_all = trainer.predict(tokenized_dataset["test"])
prediction_values = predictions_all.predictions
prediction_metrics = predictions_all.metrics

print(f"Prediction metrics on the test data:")
prediction_metrics

In [ ]:
predictions_all

### Predicted logits -> predictions probabilities with torch.softmax -> predicted labels

In [ ]:
import torch
from sklearn.metrics import accuracy_score

# softmax gets all values to be between 0 & 1 and all the values sum to 1
# this is known as "prediction probability",as in the model is assigning this value to how "likely" the prediction is given the sample


pred_probs = torch.softmax(torch.tensor(prediction_values),dim=0)

pred_labels = torch.argmax(pred_probs,dim=1)

true_lables = tokenized_dataset["test"]["label"]

test_accuracy = accuracy_score(true_lables,pred_labels)

print(f"Test accuracy: {test_accuracy}")


## Making and inspecting predictions on custom text data

In [ ]:
local_model_path = "models/learn-hf-food-not_food-text-classification-model-distilbert-base-uncased"

huggingface_model_path = "yjenny2512/learn-hf-food-not_food-text-classification-model-distilbert-base-uncased"



### Discussing ways to make predictions(inference)

* Note : Whenever you hear the word "inference" it means to use a model to make predictions on data

Two main ways to perform inference:

1. **Pipeline** mode - using `transformers.pipeline` to load our model and perform text classifications

2. **PyTorch** mode - using a combination of `transformers.AutoTokenizer` and `transformers.AutoModelForSequenceClassification` and passing each our target model name

Each mode supports:
1. Predictions one at a time(fast but can be slower with many samples)
2. Batches of predictions at a time(faster but up to a point)

In [ ]:
# Setup our device for making predictions

def set_device():
  if torch.cuda.is_available():
    device = torch.device("cuda")
  elif torch.backends.mps.is_available():
    device = torch.device("mps")
  else:
    device = torch.device("cpu")
  return device

device = set_device()
print(f"using device:{device}")
device

### Making predictions with pipeline mode

In [ ]:
import torch
from transformers import pipeline

#Set the batch size
BATCH_SIZE = 32

# Create an instance of transformers.pipeline
food_not_food_classifier = pipeline(task="text-classification",
                                    model=local_model_path,
                                    device=device,
                                    top_k=1,batch_size=BATCH_SIZE
                                    )
food_not_food_classifier

In [ ]:
test_custom_sentence = "The Gate by Pete2453"
food_not_food_classifier(test_custom_sentence)

# Use pipeline with a model from Huggingface


In [ ]:
del pipeline

In [ ]:
from transformers import pipeline
food_not_food_classifier = pipeline(task="text-classification",
                                    model=huggingface_model_path,
                                    device=device,
                                    top_k=1,
                                    batch_size=BATCH_SIZE)
food_not_food_classifier(test_custom_sentence
                         )

### Making multiple predictions at the same time with batch prediction


In [ ]:
# Create a list of sentences to make predictions on
sentences = [
    "I whipped up a fresh batch of code, but it seems to have a syntax error.",
    "We need to marinate these ideas overnight before presenting them to the client.",
    "The new software is definitely a spicy upgrade, taking some time to get used to.",
    "Her social media post was the perfect recipe for a viral sensation.",
    "He served up a rebuttal full of facts, leaving his opponent speechless.",
    "The team needs to simmer down a bit before tackling the next challenge.",
    "The presentation was a delicious blend of humor and information, keeping the audience engaged.",
    "A beautiful array of fake wax foods (shokuhin sampuru) in the front of a Japanese restaurant.",
    "Daniel Bourke is really cool :D",
    "My favoruite food is biltong!"
]

food_not_food_classifier(sentences)

### Time our model across larger sample sizes


In [ ]:
import time

# Create 1000 sentences

sentences_1000 = sentences * 100

len(sentences_1000)

# Time how long it takes to make predictions on all sentences (one at a time)
print(f"Number of sentences: {len(sentences_1000)}")
start_time_one_at_a_time = time.time()
for sentence in sentences_1000:
  food_not_food_classifier(sentence)
end_time_one_at_a_time = time.time()

total_time_one_at_a_time = end_time_one_at_a_time - start_time_one_at_a_time
print(f"Total time to make predictions one at a time: {total_time_one_at_a_time}")
print(f"Time per prediction: {total_time_one_at_a_time / len(sentences_1000)}")


In [ ]:
# Let's now use pipeline in batches
for i in [10,100,1000]:
  sentences_big = sentences*i
  print(f"Number of sentences: {len(sentences_big)}")
  start_time_batches = time.time()
  food_not_food_classifier(sentences_big)
  end_time_batches = time.time()

  total_time_batches = end_time_batches - start_time_batches
  print(f"Total time to make predictions in batches: {total_time_batches}")
  print(f"Time per prediction: {total_time_batches / len(sentences_big)}")

### Making predictions with PyTorch

Steps with PyTorch predictions:
1. Create the tokenizer with `AutoTokenizer`
2. Create the model with `AutoModel`(AutoModelForSequenceClassification)
3. Tokenize text with 1
4. Make predictions with 2
5. Format prediction

In [ ]:
from transformers import AutoTokenizer

# Setup the model path
model_path = "yjenny2512/learn-hf-food-not_food-text-classification-model-distilbert-base-uncased"

# Create an example to predict on
sample_food_text = "A delicious photo of a plate oft scrambled eggs, toast and bacon"

# Prepare the tokenizer
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_path)
inputs = tokenizer(sample_food_text,
                   return_tensors="pt") # pt stands for PYTorch
inputs

In [ ]:
from transformers import AutoModelForSequenceClassification
# Load our text classification model

model = AutoModelForSequenceClassification.from_pretrained(pretrained_model_name_or_path=model_path)
model

In [ ]:
import torch

with torch.inference_mode():
  outputs = model(**inputs) # ** means input all of the dictionary keys as named arguments
 # outputs = model(input_ids=inputs["input_ids"],
  #                attention_mask=inputs["attentiin_mask"])
outputs


In [ ]:
model.config.id2label

In [ ]:
# Convert logits to prediction prabability + label
predicted_class_id = outputs.logits.argmax().item()
prediction_probability = torch.softmax(outputs.logits,dim=1).max().item()

print(f"Text:{sample_food_text}")
print(f"Predicted label: {model.config.id2label[predicted_class_id]}")
print(f"Prediction probability: {prediction_probability}")

## Turning our model into a demo

We're using Gradio for this

### Creating a function to perform inference

1. Take an input of string
2. Setup a text classification pipeline
3. Get the output from the pipeline
4. Return the output from the pipeline in step 3 as a formatted dictionary with the format:
{"label_1": probability_1,"label_2":probability_2}



In [ ]:
from typing import Dict

def food_not_food_classifier(text:str)-> Dict[str,float]:
  food_not_food_classifier_pipeline = pipeline(task="text-classification",
                                      model=local_model_path,
                                      batch_size=32,
                                      device="cuda" if torch.cuda.is_available() else "cpu",
                                      top_k=None) # return all possible values
  outputs = food_not_food_classifier_pipeline(text)[0]

  output_dict = {}
  for item in outputs :
    output_dict[item["label"]] = item["score"]


  return output_dict

food_not_food_classifier("Yo we're building a local demo")

### Build a small gradio demo to run locally

1. Import Gradio
2. Create a gradio interface - https://www.gradio.app/docs/gradio/interface
3. Launch the interface




In [ ]:
import gradio as gr
demo = gr.Interface(
    fn = food_not_food_classifier,
    inputs="text",
    outputs = gr.Label(num_top_classes=2),
    title = "Food Not Food Classifier",
    description="A text classifier to determin eif a sentence is about food or not",
    examples = [["I whipped up a fresh batch of code, but it seems to have a syntax error"],
                ["A plate of pancakes and strawberry icing"]]
)

demo.launch()

## Making our demo publicly accessible

There are 2 mai ways to make it publicly accessible with HF Spaces :
1. Manually
2. Programmatically

To create a Space programmatically we're going to need three files:
1. `app.py` - this is the main app functionality of our demo
2. `requirements.txt` - these are the dependencies which our app wsill require
3. `README.md` - this will explain what our demo is about and will add some metadata in YAML format

To create this we'll use the following structure:

```

demos/
└── food_not_food_text_classifier/
    ├── app.py
    ├── README.md
    └── requirements.txt


```

### Making a directory to store our demo

In [ ]:
from pathlib import Path

# Make directory for demos
demos_dir = Path("../demos")
demos_dir.mkdir(exist_ok=True)

# Create a folder for the food_not_food_text_classifier demo
food_not_food_classifier_demo_dir = Path(demos_dir,"food_not_food_text_classifier")
food_not_food_classifier_demo_dir.mkdir(exist_ok=True)

### Making an app.py file

It will contain the main logic of our appplacation to run

When we upload it to Hugging face Spaces,Spaces will try to run `app.py` automatically

In our `app.py` file we want to:
1. Import packages
2. Define our function to use our model(this will work with Gradio)
3. Create a demo with Gradio
4. Run the demo with demo.launch()

To create each of the files we're going to use the magic command `%%writefile`

In [ ]:
%%writefile ../demos/food_not_food_text_classifier/app.py

import torch
import gradio as gr

from typing import Dict
from transformers import pipeline

def food_not_food_classifier(text:str)-> Dict[str,float]:
  food_not_food_classifier_pipeline = pipeline(task="text-classification",
                                      model="yjenny2512/learn-hf-food-not_food-text-classification-model-distilbert-base-uncased",
                                      batch_size=32,
                                      device="cuda" if torch.cuda.is_available() else "cpu",
                                      top_k=None) # return all possible values
  outputs = food_not_food_classifier_pipeline(text)[0]

  output_dict = {}
  for item in outputs :
    output_dict[item["label"]] = item["score"]


  return output_dict

description = """
A text classifier to determine if a sentence is about food or not food

Fine-tuned from [Distilbert](https://huggingface.co/distilbert/distilbert-base-uncased) a dataset of LLM generated food/not_food image captions

See [source code](https://github.com/yjenny2512/learn-hf-food-not_food-text-classification-model-distilbert-base-uncased)
"""

demo = gr.Interface(
    fn = food_not_food_classifier,
    inputs="text",
    outputs = gr.Label(num_top_classes=2),
    title = "🥑🚫🍗Food Not Food Classifier",
    description= description,
    examples = [["I whipped up a fresh batch of code, but it seems to have a syntax error"],
                ["A plate of pancakes and strawberry icing"]] )


if __name__ == "__main__":
  demo.launch()

### Making a README file

This is in markdown fomat

With special YAML block at the top

The YAML block at the top is used for metadata + settings



In [ ]:
%%writefile ../demos/food_not_food_text_classifier/README.md
---
title: Food Not Food Text Classifier
emoji: 🍗🚫🥑
colorFrom: blue
colorTo: yellow
sdk: gradio
app_file: app.py
pinned: false
license: apache-2.0
---

# 🍗🚫🥑 Food Not Food Text Classifier

Small demo to showcase a text classifier to determine if a sentence is about food or not food.

DistilBERT model fine-tuned on a small synthetic dataset of 250 generated food/not_food image captions.

See [source code notebook](https://github.com/yjenny2512/learn-hf-food-not_food-text-classification-model-distilbert-base-uncased)


### Making a requirements file

This file is going to tell Hugging Face Space which versions/packages to use



In [ ]:
%%writefile ../demos/food_not_food_text_classifier/requirements.txt
gradio
torch
transformers

### Uploading our demo to Hugging Face Spaces

To do so we use the Hugging Face Hub Python API

To get our demo on HF Spaces, we can do the following :

1. Import necessary functions
2. Define what we want to upload
3. Create a repo:https://huggingface.co/docs/huggingface_hub/guides/repository
4. Get the name of our repo from the upload
5. Upload the contents of our `../demos/fodd_not_food_text_classifier/` to our Hugging Face Hub repo
6. Inspect the results

In [ ]:

#1. Import the required methods for uploading to the HF hub
from huggingface_hub import (
    create_repo,
    get_full_repo_name,
    upload_file,
    upload_folder
)

# 2. Define the parameters we'd like to use for uploading our Space
LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD = "../demos/food_not_food_text_classifier"
HF_TARGET_SPACE_NAME = "learn_hf_food_not_food_text_classifier_demo"
HF_REPO_TYPE = "space"
HF_SPACE_SDK = "gradio"

#3. Create a space repo on Hugging Face Hub
print(f"Creating repo on HF Hub with name: {HF_TARGET_SPACE_NAME}")
repo_id = create_repo(
    repo_id=HF_TARGET_SPACE_NAME,
    repo_type=HF_REPO_TYPE,
    private=False,
    space_sdk=HF_SPACE_SDK,
    exist_ok=True,
)

#4. Get the full repo name
hf_full_repo_name = get_full_repo_name(model_id=HF_TARGET_SPACE_NAME)
print(f"Repo name: {hf_full_repo_name}")

#5. Upload our demo folder
print(f"Uploading {LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD} to repo {hf_full_repo_name}")
folder_upload_url = upload_folder(
    repo_id=hf_full_repo_name,
    folder_path=LOCAL_DEMO_FOLDER_PATH_TO_UPLOAD,
    path_in_repo=".",
    repo_type=HF_REPO_TYPE,
    commit_message="Uploading our food not food text classifier demo from a notebook!!!",
    )

print(f"Demo folder successfully uploladed with commit URL: {folder_upload_url}")